In [30]:
import polars as pl
import os

from pathlib import Path

In [31]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/18_Pairwise Ranking HepG2 REMC/')
DATASET_PATH = WORKING_PATH / 'dataset'
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined'

# Merge the experiment results

In [32]:
# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [33]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_aucprc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",12,75.4,0.84,74.9,0.8437,0.8427,0.7218,0.7745,0.7472,0.922
"""LogisticRegression""",1011,"""H3K4me3""",null,75.2,0.8226,76.6,0.8425,0.8284,0.7966,0.6868,0.7377,0.797
"""RandomForest""",1011,"""H3K4me3""",null,75.9,0.8281,77.2,0.8531,0.8381,0.7898,0.714,0.75,0.788
"""SVM_Linear""",1011,"""H3K4me3""",null,75.1,0.8231,76.8,0.8428,0.8286,0.7976,0.691,0.7405,0.794
"""DirectRanker""",123,"""H3K4me3""",14,77.1,0.86,73.0,0.8044,0.7919,0.7097,0.7857,0.7458,0.922
…,…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,75.2,0.8224,75.9,0.8204,0.8165,0.7665,0.7337,0.7497,0.759
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",54,79.2,0.87,76.4,0.8592,0.8664,0.7436,0.7835,0.7631,0.989
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,73.4,0.8155,72.9,0.8125,0.7815,0.7238,0.7134,0.7186,0.777


In [34]:
# Save the results by model 
MODEL_RESULT_FOLDER = OUTPUT_PATH / 'model_results'
os.makedirs(MODEL_RESULT_FOLDER, exist_ok=True)

for model_name in pl_df["model"].unique():
    pl_df.filter(pl_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}.csv", include_header=True)

In [35]:
# Save the results for all models and all seeds
pl_df.write_csv(OUTPUT_PATH / "HepG2 ENCSR134DWG_all_results.csv", include_header=True)

In [36]:
summary_df = (
    pl_df
    .group_by(["histone_marker", "model"])
    .agg([
        pl.col("val_accuracy").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy").std().alias("val_accuracy_std"),
        pl.col("val_auc").mean().alias("val_auc_mean"),
        pl.col("val_auc").std().alias("val_auc_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
        pl.col("test_auc").mean().alias("test_auc_mean"),
        pl.col("test_auc").std().alias("test_auc_std"),
        pl.col("test_aucprc").mean().alias("test_aucprc_mean"),
        pl.col("test_aucprc").std().alias("test_aucprc_std"),
        pl.col("antisymmetry").mean().alias("antisymmetry_mean"),
        pl.col("antisymmetry").std().alias("antisymmetry_std")
    ])
    .sort(["histone_marker", "test_accuracy_mean"], descending=[False, True])
    .with_columns(pl.col(pl.Float64).round(4))
)

In [37]:
summary_df

histone_marker,model,val_accuracy_mean,val_accuracy_std,val_auc_mean,val_auc_std,test_accuracy_mean,test_accuracy_std,test_auc_mean,test_auc_std,test_aucprc_mean,test_aucprc_std,antisymmetry_mean,antisymmetry_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""H3K27ac""","""RandomForest""",77.06,1.1739,0.8489,0.0144,75.74,1.4673,0.8446,0.0143,0.8319,0.0087,0.724,0.0157
"""H3K27ac""","""SVM_Linear""",75.04,1.417,0.8101,0.0158,73.18,1.4464,0.8009,0.0134,0.7725,0.0187,0.6894,0.0105
"""H3K27ac""","""LogisticRegression""",75.16,1.1696,0.8109,0.0157,73.18,1.3161,0.7987,0.0141,0.7715,0.0204,0.6924,0.0093
"""H3K27ac""","""DirectRanker""",73.1,2.1378,0.834,0.0152,71.96,1.6965,0.8254,0.014,0.8056,0.0171,0.7822,0.0102
"""H3K27ac-H3K27me3""","""RandomForest""",77.14,1.6772,0.8497,0.015,74.38,1.1345,0.8412,0.0121,0.8379,0.0079,0.7886,0.0234
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""H3K9me3-H3K27ac-H3K27me3""","""SVM_Linear""",71.62,1.3755,0.7786,0.0243,70.68,1.7922,0.7751,0.0179,0.7443,0.0231,0.754,0.0189
"""H3K9me3-H3K27me3""","""RandomForest""",67.38,1.3664,0.734,0.0211,65.38,2.3339,0.7163,0.026,0.6947,0.0166,0.8124,0.0125
"""H3K9me3-H3K27me3""","""DirectRanker""",66.0,1.4422,0.726,0.023,65.06,2.2188,0.7139,0.0242,0.6945,0.0207,0.9586,0.0026


In [38]:
summary_df.write_csv(OUTPUT_PATH/ f"HepG2 ENCSR134DWG Ranking.csv", include_header=True)

In [39]:
for model_name in summary_df["model"].unique():
    summary_df.filter(summary_df["model"] == model_name).write_csv(MODEL_RESULT_FOLDER / f"{model_name}_summary.csv", include_header=True)

# Merge the label distribution

In [40]:
# Read all label distribution CSV into single dataframe
label_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-label-distribution.csv")

In [41]:
label_df

seed,histone_marker,split,label,count,total,percentage
i64,str,str,i64,i64,i64,f64
1011,"""H3K4me3""","""Train""",0,4140,8000,51.75
1011,"""H3K4me3""","""Train""",1,3860,8000,48.25
1011,"""H3K4me3""","""Val""",0,553,1000,55.3
1011,"""H3K4me3""","""Val""",1,447,1000,44.7
1011,"""H3K4me3""","""Test""",0,521,1000,52.1
…,…,…,…,…,…,…
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Train""",1,3903,8000,48.79
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",0,536,1000,53.6
789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…","""Val""",1,464,1000,46.4


In [42]:
# Make the summary by seed, split and label
label_summary_df = (
    label_df
    .group_by(['split', 'label'])
    .agg(pl.col('percentage').mean().round(2).alias('average_percentage_%'))
    .sort(['split', 'label'])
)

In [43]:
label_summary_df

split,label,average_percentage_%
str,i64,f64
"""Test""",0,50.98
"""Test""",1,49.02
"""Train""",0,51.59
"""Train""",1,48.41
"""Val""",0,53.3
"""Val""",1,46.7
